# FinExpert: export the v3 adapter to GGUF for the CPU demo

1. Load the **exact adapter scored on the test sets** (`v2-pilot-seed42`) and merge it into the 16-bit base model.
2. Convert to a 4-bit **GGUF** (`Q4_K_M`), the format `llama.cpp` runs on ordinary CPUs.
3. **Check the conversion:** run the GGUF on 40 fixed validation questions and compare with the adapter's own recorded answers.
   Merging into 16-bit weights and re-quantising can change answers slightly; this measures by how much instead of assuming.
4. Upload the GGUF with a model card to Hugging Face.

**Kaggle setup:** GPU T4 x2 · Internet on · Add-ons → Secrets → `HF_TOKEN` attached · **+ Add Input → Your Work → `notebookf63e59d067`** (the run that trained `v2-pilot-seed42`).

In [ ]:
ADAPTER_RUN = "v2-pilot-seed42"
BASE_MODEL  = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
QUANT       = "q4_k_m"
GGUF_REPO   = "ManideepVarma003/finexpert-qwen3-4b-gguf"
GGUF_NAME   = "finexpert-qwen3-4b-Q4_K_M.gguf"
LORA_REPO   = "ManideepVarma003/finexpert-qwen3-4b-lora"
REPO_URL    = "https://github.com/mani1234765/finexpert.git"
CHECK_PER_SOURCE = 20          # validation questions per source for the conversion check (0 = skip)

In [ ]:
import os, sys, subprocess, glob, json, time, shutil
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
def sh(*cmd, cwd=None):
    r = subprocess.run(list(cmd), capture_output=True, text=True, cwd=cwd)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:]); raise RuntimeError(" ".join(cmd))
    return r.stdout.strip()
sh(sys.executable, "-m", "pip", "install", "-q", "unsloth")
found = glob.glob(f"/kaggle/input/**/{ADAPTER_RUN}/adapter/adapter_config.json", recursive=True)
assert len(found) == 1, f"attach the notebook that trained {ADAPTER_RUN}; found {found}"
ADAPTER_DIR = os.path.dirname(found[0])
cfg = json.load(open(found[0]))
assert cfg["base_model_name_or_path"] == BASE_MODEL, cfg["base_model_name_or_path"]
print("Adapter:", ADAPTER_DIR, "| r", cfg["r"], "alpha", cfg["lora_alpha"])

In [ ]:
# Merge into the 16-bit base and convert to GGUF (Unsloth downloads the 16-bit base and builds llama.cpp; ~15-25 min)
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(model_name=ADAPTER_DIR, max_seq_length=4096, load_in_4bit=True)
OUT = "/tmp/finexpert_gguf"
t0 = time.time()
model.save_pretrained_gguf(OUT, tokenizer, quantization_method=QUANT)
candidates = [p for p in glob.glob(f"{OUT}/**/*.gguf", recursive=True) + glob.glob("*.gguf")
              if QUANT.lower() in os.path.basename(p).lower()]
assert len(candidates) >= 1, f"no {QUANT} GGUF found: {glob.glob(f'{OUT}/**/*', recursive=True)}"
GGUF_PATH = os.path.join("/tmp", GGUF_NAME)
shutil.move(candidates[0], GGUF_PATH)
print(f"GGUF ready in {(time.time() - t0) / 60:.1f} min:", GGUF_PATH, f"{os.path.getsize(GGUF_PATH) / 1e9:.2f} GB")
del model; import gc, torch; gc.collect(); torch.cuda.empty_cache()

## Conversion check
The repository is cloned only for its data, scorers and the adapter's recorded validation answers
(`experiments/v2-pilot-seed42/finetuned_qa_predictions.jsonl`). The GGUF answers the first 20 FinQA and 20 TAT-QA questions of the fixed
validation subset (sorted ids) on CPU with greedy decoding, using the same prompt builder as training.

In [ ]:
CHECK = {}
if CHECK_PER_SOURCE:
    sh(sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python==0.3.30")   # CPU build, ~10 min
    REPO = "/tmp/finexpert-repo"
    if not os.path.exists(REPO):
        sh("git", "clone", "-q", REPO_URL, REPO)
    audit = json.load(open(f"{REPO}/data/external/audit.json"))
    for name, folder, url in [("finqa", "FinQA", "https://github.com/czyssrs/FinQA.git"),
                              ("tatqa", "TAT-QA", "https://github.com/NExTplusplus/TAT-QA.git")]:
        dest = f"{REPO}/data/external/raw/{folder}"
        if not os.path.exists(dest):
            sh("git", "clone", "-q", url, dest)
        sh("git", "-C", dest, "checkout", "-q", audit["sources"][name]["source_commit"])
    sh(sys.executable, "scripts/build_external_dataset.py", cwd=REPO)
    sys.path.insert(0, f"{REPO}/src")
    from finexpert.evaluation.prompting import build_messages, to_chatml
    from finexpert.evaluation.qa.report import score_record
    from llama_cpp import Llama

    subset = json.load(open(f"{REPO}/data/external/eval_subset_v1.json"))
    recorded = {json.loads(l)["example_id"]: json.loads(l)["prediction"]
                for l in open(f"{REPO}/experiments/{ADAPTER_RUN}/finetuned_qa_predictions.jsonl")}
    records = []
    for source in ("finqa", "tatqa"):
        wanted = sorted(subset["sources"][source]["example_ids"])[:CHECK_PER_SOURCE]
        by_id = {json.loads(l)["example_id"]: json.loads(l) for l in open(f"{REPO}/data/external/{source}/validation.jsonl")}
        records += [by_id[i] for i in wanted]

    llm = Llama(model_path=GGUF_PATH, n_ctx=4096, n_threads=os.cpu_count(), verbose=False)
    tatqa_dir = f"{REPO}/data/external/raw/TAT-QA"
    rows, t0 = [], time.time()
    for r in records:
        out = llm.create_completion(to_chatml(build_messages(r)), max_tokens=384, temperature=0.0, stop=["<|im_end|>"])
        gguf_text = out["choices"][0]["text"].strip()
        g = score_record(r, gguf_text, tatqa_dir)["strict"]
        a = score_record(r, recorded[r["example_id"]], tatqa_dir)["strict"]
        rows.append({"id": r["example_id"], "source": r["source_type"], "same_answer": g["extracted"] == a["extracted"],
                     "gguf_correct": g["correct"], "adapter_correct": a["correct"], "tokens": out["usage"]["completion_tokens"]})
    secs = (time.time() - t0) / len(rows)
    CHECK = {"questions": len(rows), "same_final_answer": sum(x["same_answer"] for x in rows),
             "gguf_correct": sum(x["gguf_correct"] for x in rows), "adapter_correct": sum(x["adapter_correct"] for x in rows),
             "seconds_per_answer_cpu": round(secs, 1), "cpu_threads": os.cpu_count()}
    json.dump(rows, open("/kaggle/working/gguf_check_details.json", "w"), indent=2)
    print(json.dumps(CHECK, indent=2))

In [ ]:
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN"))
check_text = ("Not run." if not CHECK else
    f"On {CHECK['questions']} fixed validation questions (20 FinQA + 20 TAT-QA), the GGUF gave the same final answer as the "
    f"evaluated adapter on **{CHECK['same_final_answer']}/{CHECK['questions']}**; correct answers: GGUF "
    f"{CHECK['gguf_correct']}/{CHECK['questions']} vs adapter {CHECK['adapter_correct']}/{CHECK['questions']}. "
    f"CPU speed on Kaggle ({CHECK['cpu_threads']} threads): about {CHECK['seconds_per_answer_cpu']} s per answer.")
card = f"""---
base_model: {BASE_MODEL}
license: apache-2.0
tags: [gguf, llama.cpp, finance, qlora, finqa, tat-qa]
---

# FinExpert, GGUF ({QUANT.upper()})

The [FinExpert LoRA adapter](https://huggingface.co/{LORA_REPO}) merged into Qwen3-4B-Instruct-2507 and quantised to
{QUANT.upper()} for CPU inference with llama.cpp. Evaluation, intended use and limitations: see the adapter's model card;
code and experiment log: <https://github.com/mani1234765/finexpert>.

**Conversion check.** {check_text}
Reported benchmark results belong to the adapter on the 4-bit base; this file is a convenience conversion for the demo.

**Prompt format:** ChatML with the FinExpert system prompt; the exact builder is in the adapter's quick start.
Use greedy decoding (temperature 0) and stop at `<|im_end|>`.
"""
api = HfApi()
api.create_repo(GGUF_REPO, exist_ok=True)
api.upload_file(path_or_fileobj=GGUF_PATH, path_in_repo=GGUF_NAME, repo_id=GGUF_REPO, commit_message="FinExpert Q4_K_M GGUF")
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=GGUF_REPO, commit_message="Model card")
json.dump({"adapter_run": ADAPTER_RUN, "adapter_dir": ADAPTER_DIR, "quant": QUANT, "gguf": GGUF_NAME,
           "size_gb": round(os.path.getsize(GGUF_PATH) / 1e9, 2), "check": CHECK},
          open("/kaggle/working/gguf_export_manifest.json", "w"), indent=2)
print("Published: https://huggingface.co/" + GGUF_REPO)